# Exercises XP: Open-Source LLM Strategy
Guided notebook with every TODO filled. **No GPU required.**

To avoid copying numbers by hand, the facts are pulled from **live sources** where possible:
- license and size metadata from the **Hugging Face Hub API**;
- benchmark scores (HellaSwag, MMLU, GSM8K, ARC…) from the archived **Open LLM Leaderboard (v1)** results dataset, `open-llm-leaderboard-old/contents`;
- the hardware audit from **the machine running the notebook**;
- the deployment test, which actually runs a **7B model** on CPU with llama.cpp (4-bit GGUF).

## What you'll learn
- Identify degrees of openness in LLMs and what each enables.
- Understand and compare open-source licenses.
- Assess strengths/trade-offs of open LLMs for specific constraints (CPU, math, multilingual).
- Evaluate which models fit given hardware and licensing needs.
- Use open tools/leaderboards to guide model selection.
- Plan deployment trade-offs (local vs. cloud).

In [1]:
!pip install -q huggingface_hub pandas pyarrow psutil
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [2]:
import os, json, platform, shutil, subprocess, time, re
import pandas as pd
from huggingface_hub import HfApi
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
api = HfApi()

def hub_info(repo_id):
    # license, size, gating and language tags straight from the Hugging Face Hub
    i = api.model_info(repo_id, expand=["cardData", "safetensors", "gated", "tags", "downloads"])
    card = i.card_data.to_dict() if i.card_data else {}
    langs = card.get("language") or []
    return {"model": repo_id, "url": f"https://huggingface.co/{repo_id}",
            "license": card.get("license") or next((t.split(":", 1)[1] for t in i.tags if t.startswith("license:")), "?"),
            "license_name": card.get("license_name"),
            "params_B": round(i.safetensors.total / 1e9, 2) if i.safetensors else None,
            "gated": i.gated, "n_languages_tagged": len(langs) if isinstance(langs, list) else 1,
            "downloads_30d": i.downloads}

## Exercise 1: Open Source Levels Reflection
**Use:** Fully Open, Weights Released, Architecture Only; model components; fine-tuning and domain adaptation; healthcare compliance.

In [3]:
open_source_levels = {
    "Fully Open": {
        "definition": "Everything needed to reproduce and change the model is public: architecture/training code, "
                      "pretrained weights, and (ideally) training data recipes and evaluation code, under an open license "
                      "(e.g. OLMo, Pythia, BLOOM).",
        "what_is_open": ["Source code (architecture, training & inference scripts)", "Pretrained weights (+ data/recipes)"],
        "what_you_can_do": ["Inspect internals, audit the training data and reproduce training from scratch",
                            "Modify the architecture, retrain or fully fine-tune, self-host anywhere"],
        "what_you_cannot_do": ["Ignore the license terms (attribution / copyleft / use restrictions still apply)",
                               "Avoid the compute cost: retraining from scratch remains very expensive"],
    },
    "Weights Released": {
        "definition": "The trained parameters are downloadable (often with inference code), but the training code, "
                      "data and full recipe are not public (e.g. Llama 2/3, Mistral 7B, Gemma).",
        "what_is_open": ["Pretrained (and often instruction-tuned) weights", "Model card + inference code / config"],
        "what_you_can_do": ["Run locally, quantize, and fine-tune (full, LoRA/QLoRA) for a domain",
                            "Deploy behind your own firewall within the license terms"],
        "what_you_cannot_do": ["Reproduce or audit the original training data and process",
                               "Always use commercially without conditions (custom licenses: user caps, AUP, attribution)"],
    },
    "Architecture Only": {
        "definition": "Only the model design is published (paper / code of the network), without trained weights "
                      "(e.g. the original Transformer paper, or code released without checkpoints).",
        "what_is_open": ["Architecture description / model code", "Hyper-parameters and design choices (sometimes)"],
        "what_you_can_do": ["Understand, implement and study the design", "Train your own model from scratch with your data"],
        "what_you_cannot_do": ["Use a ready pretrained model or fine-tune it: there are no weights",
                               "Get comparable quality without massive data and GPU budgets"],
    },
}
pd.DataFrame(open_source_levels).T

,definition,what_is_open,what_you_can_do,what_you_cannot_do
Fully Open,"Everything needed to reproduce and change the model is public: architecture/training code, pretrained weights, and (...","[Source code (architecture, training & inference scripts), Pretrained weights (+ data/recipes)]","[Inspect internals, audit the training data and reproduce training from scratch, Modify the architecture, retrain or...","[Ignore the license terms (attribution / copyleft / use restrictions still apply), Avoid the compute cost: retrainin..."
Weights Released,"The trained parameters are downloadable (often with inference code), but the training code, data and full recipe are...","[Pretrained (and often instruction-tuned) weights, Model card + inference code / config]","[Run locally, quantize, and fine-tune (full, LoRA/QLoRA) for a domain, Deploy behind your own firewall within the li...","[Reproduce or audit the original training data and process, Always use commercially without conditions (custom licen..."
Architecture Only,"Only the model design is published (paper / code of the network), without trained weights (e.g. the original Transfo...","[Architecture description / model code, Hyper-parameters and design choices (sometimes)]","[Understand, implement and study the design, Train your own model from scratch with your data]","[Use a ready pretrained model or fine-tune it: there are no weights, Get comparable quality without massive data and..."


In [4]:
comparison_table = """| Openness level | What's open? | Impact on retraining/modifying |
| --- | --- | --- |
| Fully Open | Code + weights (+ data/recipes) | You can inspect, modify and retrain the entire model, and reproduce or audit training. |
| Weights Released | Trained weights + inference code (no full training code/data) | You can fine-tune and adapt (LoRA, full FT) and quantize, but cannot reproduce or audit the original training. |
| Architecture Only | Model design / code, no weights | You know the structure but must pretrain from scratch, so domain adaptation is only possible with huge compute. |"""
display(Markdown(comparison_table))

| Openness level | What's open? | Impact on retraining/modifying |
| --- | --- | --- |
| Fully Open | Code + weights (+ data/recipes) | You can inspect, modify and retrain the entire model, and reproduce or audit training. |
| Weights Released | Trained weights + inference code (no full training code/data) | You can fine-tune and adapt (LoRA, full FT) and quantize, but cannot reproduce or audit the original training. |
| Architecture Only | Model design / code, no weights | You know the structure but must pretrain from scratch, so domain adaptation is only possible with huge compute. |

In [5]:
comparative_paragraph = """A fully open model publishes its code, weights and, ideally, its training data recipe, so it can be audited, modified and retrained end to end; its main advantage is transparency and reproducibility, but retraining it is still expensive. A weights-released model gives you the trained parameters (usually with inference code) but not the full training pipeline or data: it is the most practical option for fine-tuning and self-hosting, but you cannot verify how it was trained and its custom license may restrict commercial use. An architecture-only release describes the network without any trained weights, which is useful for research and for understanding the design, but its big limitation is that you must pretrain from scratch, at a cost out of reach for most teams."""
print(comparative_paragraph)

A fully open model publishes its code, weights and, ideally, its training data recipe, so it can be audited, modified and retrained end to end; its main advantage is transparency and reproducibility, but retraining it is still expensive. A weights-released model gives you the trained parameters (usually with inference code) but not the full training pipeline or data: it is the most practical option for fine-tuning and self-hosting, but you cannot verify how it was trained and its custom license may restrict commercial use. An architecture-only release describes the network without any trained weights, which is useful for research and for understanding the design, but its big limitation is that you must pretrain from scratch, at a cost out of reach for most teams.


In [6]:
healthcare_prompt_answer = """At least the "Weights Released" level is essential (and "Fully Open" is preferable): only with the pretrained weights can we fine-tune the model on clinical data inside our own secure, HIPAA/GDPR-compliant infrastructure without sending patient data to a third party, and full openness additionally lets us audit the training data and behaviour that regulators and clinicians will ask about."""
print(healthcare_prompt_answer)

At least the "Weights Released" level is essential (and "Fully Open" is preferable): only with the pretrained weights can we fine-tune the model on clinical data inside our own secure, HIPAA/GDPR-compliant infrastructure without sending patient data to a third party, and full openness additionally lets us audit the training data and behaviour that regulators and clinicians will ask about.


## Exercise 2: License Check for SaaS Use
**Use:** HF model pages; permissive vs. copyleft; commercial clauses; restrictions (MAU caps, attribution, export controls).

The two models are `mistralai/Mistral-7B-Instruct-v0.3` and `meta-llama/Llama-2-7b-chat-hf`. Their license fields are read live from the Hub below. The license terms in the checklist come from the license texts (Apache 2.0 and the Llama 2 Community License Agreement + Acceptable Use Policy).

In [7]:
hub_rows = [hub_info(m) for m in ["mistralai/Mistral-7B-Instruct-v0.3", "meta-llama/Llama-2-7b-chat-hf"]]
pd.DataFrame(hub_rows)

,model,url,license,license_name,params_B,gated,n_languages_tagged,downloads_30d
0,mistralai/Mistral-7B-Instruct-v0.3,https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3,apache-2.0,None,7.25,False,0,2062564
1,meta-llama/Llama-2-7b-chat-hf,https://huggingface.co/meta-llama/Llama-2-7b-chat-hf,llama2,None,6.74,manual,1,356625


In [8]:
license_checklist = [
    {
        "model": "mistralai/Mistral-7B-Instruct-v0.3",
        "url": "https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3",
        "license": hub_rows[0]["license"],                                # apache-2.0 (permissive)
        "commercial_use": "Yes",
        "restrictions": [
            "Keep the Apache 2.0 license text and any NOTICE when redistributing; state significant modifications",
            "No trademark rights: you cannot use the 'Mistral' name/logo to brand your product",
            "No user caps, no field-of-use limits, no geographic restrictions (only general export-control laws apply)",
            "Patent license terminates if you sue contributors for patent infringement over the work",
        ],
    },
    {
        "model": "meta-llama/Llama-2-7b-chat-hf",
        "url": "https://huggingface.co/meta-llama/Llama-2-7b-chat-hf",
        "license": f"{hub_rows[1]['license']} (Llama 2 Community License Agreement)",
        "commercial_use": "Conditional",
        "restrictions": [
            "If your products had > 700 million monthly active users on the release date, you need a separate license from Meta",
            "Must follow Meta's Acceptable Use Policy (no illegal, harmful, deceptive uses, e.g. weapons, malware, harassment)",
            "Redistribution must include the license and the notice 'Llama 2 is licensed under the LLAMA 2 Community License, "
            "Copyright (c) Meta Platforms, Inc.' (attribution)",
            "Cannot use Llama 2 outputs to improve any other LLM (except Llama 2 derivatives)",
            "Gated download (accept the license on HF); must comply with applicable laws incl. trade/export controls",
        ],
    },
]
pd.DataFrame(license_checklist)

,model,url,license,commercial_use,restrictions
0,mistralai/Mistral-7B-Instruct-v0.3,https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3,apache-2.0,Yes,"[Keep the Apache 2.0 license text and any NOTICE when redistributing; state significant modifications, No trademark ..."
1,meta-llama/Llama-2-7b-chat-hf,https://huggingface.co/meta-llama/Llama-2-7b-chat-hf,llama2 (Llama 2 Community License Agreement),Conditional,"[If your products had > 700 million monthly active users on the release date, you need a separate license from Meta,..."


### Checklist
- [x] **mistralai/Mistral-7B-Instruct-v0.3** ([model page](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3))
  - Type of license:
    - [x] Apache 2.0 (permissive, OSI-approved)
  - Commercial use allowed:
    - [x] **Yes**, for any purpose, including SaaS, with no user limit
  - Restrictions:
    - [x] Attribution: keep the license and NOTICE files and mark modifications
    - [x] No trademark use of "Mistral"; patent licence lost if you start patent litigation
    - [x] No MAU cap, no geographic restriction (only general export-control law)
- [x] **meta-llama/Llama-2-7b-chat-hf** ([model page](https://huggingface.co/meta-llama/Llama-2-7b-chat-hf))
  - Type of license:
    - [x] Llama 2 Community License (custom, source-available, **not** OSI open source)
  - Commercial use allowed:
    - [x] **Conditional**: allowed for products under **700 M monthly active users**; above that you need Meta's permission
  - Restrictions:
    - [x] Acceptable Use Policy (prohibited uses) and gated access (accept terms on HF)
    - [x] Attribution notice + license copy when redistributing
    - [x] Outputs may not be used to train or improve other non-Llama LLMs
    - [x] Comply with applicable laws, including trade and export controls

**SaaS verdict:** both can be used in a SaaS product. Mistral (Apache 2.0) is the lowest-risk option. Llama 2 is fine for most start-ups, but legal teams must accept the AUP, the "no training other LLMs" clause and the MAU threshold.

## Exercise 3: LLM Matchmaker Challenge
**Use:** CPU inference, low-end laptops, multilingual needs; HF filters; size ≤ 7B; benchmarks.

The Open LLM Leaderboard v1 does not include BoolQ, ReClor or FLORES. We use its reasoning proxies instead: **ARC** and **MMLU** for logic and knowledge, **GSM8K** for math. For languages we use the number of languages declared on each model card (Hub API).

In [9]:
filters_by_team = {
    "LegalTech": ["https://huggingface.co/models?pipeline_tag=text-generation&library=gguf&sort=trending  (CPU-ready quantized files)",
                  "size <= 7B; tags: reasoning / long context; leaderboard sort by ARC + MMLU",
                  "license must allow commercial SaaS (apache-2.0 / mit)"],
    "EdTech": ["https://huggingface.co/models?pipeline_tag=text-generation&search=math&sort=trending",
               "size <= 4B preferred (8 GB RAM laptops), GGUF / int4 available; leaderboard sort by GSM8K"],
    "Global NGO": ["https://huggingface.co/models?pipeline_tag=text-generation&language=fr&language=es&language=ar&language=sw&language=hi&sort=trending",
                   "tag: multilingual; size <= 8B; GGUF available; count of languages on the model card"],
}
filters_by_team

{'LegalTech': ['https://huggingface.co/models?pipeline_tag=text-generation&library=gguf&sort=trending  (CPU-ready quantized files)',
  'size <= 7B; tags: reasoning / long context; leaderboard sort by ARC + MMLU',
  'license must allow commercial SaaS (apache-2.0 / mit)'],
 'EdTech': ['https://huggingface.co/models?pipeline_tag=text-generation&search=math&sort=trending',
  'size <= 4B preferred (8 GB RAM laptops), GGUF / int4 available; leaderboard sort by GSM8K'],
 'Global NGO': ['https://huggingface.co/models?pipeline_tag=text-generation&language=fr&language=es&language=ar&language=sw&language=hi&sort=trending',
  'tag: multilingual; size <= 8B; GGUF available; count of languages on the model card']}

In [10]:
import requests, io
LB_URL = "https://huggingface.co/api/datasets/open-llm-leaderboard-old/contents/parquet/default/train/0.parquet"
lb = pd.read_parquet(io.BytesIO(requests.get(LB_URL, timeout=120).content))
lb = lb.rename(columns={"#Params (B)": "params_B", "Hub License": "license", "Average ⬆️": "average"})
lb = lb[lb["Available on the hub"]].drop_duplicates("fullname")
print(f"Open LLM Leaderboard v1: {len(lb)} models with scores")
COLS = ["fullname", "params_B", "Architecture", "license", "ARC", "HellaSwag", "MMLU", "Winogrande", "GSM8K"]

def lb_rows(names):
    return lb[lb.fullname.isin(names)][COLS].set_index("fullname").loc[[n for n in names if n in set(lb.fullname)]].round(1)

Open LLM Leaderboard v1: 4969 models with scores


In [11]:
candidates_by_team = {
    "LegalTech": ["microsoft/Phi-3-mini-128k-instruct", "microsoft/Phi-3-mini-4k-instruct", "openchat/openchat-3.5-0106",
                  "mistralai/Mistral-7B-Instruct-v0.2", "meta-llama/Llama-2-7b-chat-hf"],
    "EdTech": ["microsoft/Phi-3-mini-4k-instruct", "meta-math/MetaMath-Mistral-7B", "microsoft/phi-2",
               "Qwen/Qwen1.5-1.8B-Chat", "google/gemma-2b-it"],
}
for team, names in candidates_by_team.items():
    print(f"=== {team}"); display(lb_rows(names))

=== LegalTech


,params_B,Architecture,license,ARC,HellaSwag,MMLU,Winogrande,GSM8K
fullname,,,,,,,,
microsoft/Phi-3-mini-128k-instruct,3,Unknown,mit,63.1,80.1,68.7,72.8,69.5
microsoft/Phi-3-mini-4k-instruct,3,Unknown,mit,63.0,80.6,69.1,72.4,74.5
openchat/openchat-3.5-0106,7,MistralForCausalLM,apache-2.0,66.0,82.9,65.0,81.8,68.2
mistralai/Mistral-7B-Instruct-v0.2,7,MistralForCausalLM,apache-2.0,63.1,84.9,60.8,77.2,40.0
meta-llama/Llama-2-7b-chat-hf,6,LlamaForCausalLM,llama2,52.9,78.6,48.3,71.7,7.4


=== EdTech


,params_B,Architecture,license,ARC,HellaSwag,MMLU,Winogrande,GSM8K
fullname,,,,,,,,
microsoft/Phi-3-mini-4k-instruct,3,Unknown,mit,63.0,80.6,69.1,72.4,74.5
meta-math/MetaMath-Mistral-7B,7,MistralForCausalLM,apache-2.0,60.7,82.6,62.0,75.8,68.8
microsoft/phi-2,2,PhiForCausalLM,mit,61.1,75.1,58.1,74.3,54.8
Qwen/Qwen1.5-1.8B-Chat,1,Qwen2ForCausalLM,other,38.7,60.0,45.9,59.7,19.0
google/gemma-2b-it,2,GemmaForCausalLM,gemma,43.9,62.7,37.6,60.9,5.5


In [12]:
ngo = pd.DataFrame([hub_info(m) for m in ["Qwen/Qwen2.5-7B-Instruct", "bigscience/bloomz-7b1", "CohereForAI/aya-23-8B",
                                          "facebook/nllb-200-distilled-600M", "mistralai/Mistral-7B-Instruct-v0.3"]])
ngo["leaderboard MMLU"] = ngo["model"].map(lb.set_index("fullname")["MMLU"]).round(1)
ngo

,model,url,license,license_name,params_B,gated,n_languages_tagged,downloads_30d,leaderboard MMLU
0,Qwen/Qwen2.5-7B-Instruct,https://huggingface.co/Qwen/Qwen2.5-7B-Instruct,apache-2.0,None,7.62,False,1,7631037,NaN
1,bigscience/bloomz-7b1,https://huggingface.co/bigscience/bloomz-7b1,bigscience-bloom-rail-1.0,None,7.07,False,46,6438,37.8
2,CohereForAI/aya-23-8B,https://huggingface.co/CohereForAI/aya-23-8B,cc-by-nc-4.0,None,8.03,auto,23,8312,56.3
3,facebook/nllb-200-distilled-600M,https://huggingface.co/facebook/nllb-200-distilled-600M,cc-by-nc-4.0,None,NaN,False,196,793381,NaN
4,mistralai/Mistral-7B-Instruct-v0.3,https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3,apache-2.0,None,7.25,False,0,2062564,NaN


**Notes on the NGO candidates.**
- `bloomz-7b1` covers 46 natural languages (it was trained on the ROOTS corpus), but it is weak at general reasoning (MMLU ≈ 38) and uses a RAIL license with use restrictions.
- `aya-23-8B` covers 23 languages very well, but it is CC-BY-**NC**: fine for a non-profit's internal use, though not for commercial services, and it is slightly above 7B.
- `nllb-200` translates 200 languages but is a translation model, not a chatbot.
- `Qwen2.5-7B-Instruct`: its card metadata tags only 1 language (the `n_languages_tagged` column), but the model card text states multilingual support for **29+ languages**. Tags are not always complete, so read the card. It covers Chinese, English, French, Spanish, Portuguese, German, Italian, Russian, Japanese, Korean, Vietnamese, Thai, Arabic…, and has strong general ability and an **Apache-2.0** license. A 4-bit GGUF (~4.7 GB) runs on a 16 GB laptop.

In [13]:
matchmaker_table = """| Team | Needs | Your Pick |
| --- | --- | --- |
| LegalTech | Fast model for logic-heavy chatbot on CPU | **microsoft/Phi-3-mini-128k-instruct** (3.8B, MIT: best MMLU/ARC of the ≤4B models, 4-bit GGUF ≈ 2.4 GB runs fast on CPU, 128k context for long contracts) |
| EdTech | Logic/math-focused LLM on low-end laptops | **microsoft/Phi-3-mini-4k-instruct** (GSM8K 74.5, higher than the 7B MetaMath-Mistral at 68.8, with half the RAM; MIT license) |
| Global NGO | Model that speaks 5+ languages well | **Qwen/Qwen2.5-7B-Instruct** (29+ languages incl. French, Spanish, Arabic, Chinese; Apache-2.0; 4-bit fits a 16 GB laptop). For translation-only workflows, add facebook/nllb-200-distilled-600M |"""
display(Markdown(matchmaker_table))

| Team | Needs | Your Pick |
| --- | --- | --- |
| LegalTech | Fast model for logic-heavy chatbot on CPU | **microsoft/Phi-3-mini-128k-instruct** (3.8B, MIT: best MMLU/ARC of the ≤4B models, 4-bit GGUF ≈ 2.4 GB runs fast on CPU, 128k context for long contracts) |
| EdTech | Logic/math-focused LLM on low-end laptops | **microsoft/Phi-3-mini-4k-instruct** (GSM8K 74.5, higher than the 7B MetaMath-Mistral at 68.8, with half the RAM; MIT license) |
| Global NGO | Model that speaks 5+ languages well | **Qwen/Qwen2.5-7B-Instruct** (29+ languages incl. French, Spanish, Arabic, Chinese; Apache-2.0; 4-bit fits a 16 GB laptop). For translation-only workflows, add facebook/nllb-200-distilled-600M |

## Exercise 4: Local Readiness Audit
**Use:** RAM, disk and OS; llama.cpp requirements (AVX/SSE, cmake/make/gcc/clang); quantized formats (GGUF).

The specs below are **measured on the machine running this notebook**. In this saved copy that is a 4-core Linux cloud container; on Colab you will see Colab's VM.

In [14]:
import psutil
cpu_flags = open("/proc/cpuinfo").read() if os.path.exists("/proc/cpuinfo") else ""
system_specs = {
    "ram_gb": round(psutil.virtual_memory().total / 1024**3, 1),
    "free_disk_gb": round(shutil.disk_usage("/").free / 1024**3, 1),
    "os": f"{platform.system()} {platform.release()} ({platform.machine()})",
    "cpu_cores": psutil.cpu_count(logical=True),
}
system_specs

{'ram_gb': 15.7,
 'free_disk_gb': 11.1,
 'os': 'Linux 6.18.44-fc-v114 (x86_64)',
 'cpu_cores': 4}

In [15]:
ok = lambda b: "✅" if b else "❌"
is_linux = platform.system() == "Linux" or "microsoft" in platform.release().lower()
readiness_table = f"""| Requirement | Your System Specs | Meets Requirement? |
| --- | --- | --- |
| RAM (>= 16 GB) | {system_specs['ram_gb']} GB | {ok(system_specs['ram_gb'] >= 16)} |
| Free Disk Space (>= 40 GB) | {system_specs['free_disk_gb']} GB | {ok(system_specs['free_disk_gb'] >= 40)} |
| OS (Linux/WSL2) | {system_specs['os']} | {ok(is_linux)} |"""
display(Markdown(readiness_table))

| Requirement | Your System Specs | Meets Requirement? |
| --- | --- | --- |
| RAM (>= 16 GB) | 15.7 GB | ❌ |
| Free Disk Space (>= 40 GB) | 11.1 GB | ❌ |
| OS (Linux/WSL2) | Linux 6.18.44-fc-v114 (x86_64) | ✅ |

In [16]:
flags = set(re.findall(r"\b(sse4_2|avx|avx2|avx512f|fma|f16c|neon|asimd)\b", cpu_flags))
def version(cmd):
    path = shutil.which(cmd)
    if not path:
        return "not installed"
    out = subprocess.run([cmd, "--version"], capture_output=True, text=True).stdout.splitlines()
    return out[0] if out else path

llama_cpp_readiness = {
    "cpu_instruction_support": ", ".join(sorted(flags)) or "unknown",
    "tooling": [f"cmake: {version('cmake')}", f"gcc: {version('gcc')}", f"clang: {version('clang')}", f"make: {version('make')}", f"git: {version('git')}"],
    "other_requirements": "Python 3.8+ for llama-cpp-python (prebuilt CPU wheels avoid compiling); a GGUF model file "
                          "(7B Q4_K_M ≈ 4.4 GB on disk, ≈ 5-6 GB RAM with a 2-4k context); optional BLAS/Metal/CUDA for speed",
}
llama_cpp_readiness

{'cpu_instruction_support': 'avx, avx2, avx512f, f16c, fma, sse4_2',
 'tooling': ['cmake: cmake version 3.28.3',
  'gcc: gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0',
  'clang: Ubuntu clang version 18.1.3 (1ubuntu1)',
  'make: GNU Make 4.3',
  'git: git version 2.43.0'],
 'other_requirements': 'Python 3.8+ for llama-cpp-python (prebuilt CPU wheels avoid compiling); a GGUF model file (7B Q4_K_M ≈ 4.4 GB on disk, ≈ 5-6 GB RAM with a 2-4k context); optional BLAS/Metal/CUDA for speed'}

In [17]:
upgrade_actions = []
if system_specs["ram_gb"] < 16:
    upgrade_actions.append(f"RAM {system_specs['ram_gb']} GB < 16 GB: fine for a 7B 4-bit model (~6 GB) but tight for 8-bit or "
                           "for running other apps; add RAM modules to 16-32 GB (a swap file works but is very slow, not recommended).")
if system_specs["free_disk_gb"] < 40:
    upgrade_actions.append(f"Disk {system_specs['free_disk_gb']} GB free < 40 GB: clean caches/old models (~/.cache/huggingface, pip cache) "
                           "or attach a new SSD; one 7B Q4 GGUF needs ~4.4 GB but keeping several models + FP16 originals needs 40 GB+.")
if not is_linux:
    upgrade_actions.append("OS: install a Linux distro or enable WSL2 on Windows (wsl --install).")
if not {"avx2"} & flags and not {"neon", "asimd"} & flags:
    upgrade_actions.append("CPU lacks AVX2/NEON: llama.cpp will be very slow, consider a newer CPU or cloud.")
upgrade_actions = upgrade_actions or ["No upgrade needed: this machine meets all requirements."]
upgrade_actions

['RAM 15.7 GB < 16 GB: fine for a 7B 4-bit model (~6 GB) but tight for 8-bit or for running other apps; add RAM modules to 16-32 GB (a swap file works but is very slow, not recommended).',
 'Disk 11.1 GB free < 40 GB: clean caches/old models (~/.cache/huggingface, pip cache) or attach a new SSD; one 7B Q4 GGUF needs ~4.4 GB but keeping several models + FP16 originals needs 40 GB+.']

## Exercise 5: Benchmark-Based Model Explorer
**Use:** Open LLM Leaderboard (v1: HellaSwag and MMLU scores); license types; use-case mapping.

We filter the leaderboard to ≤ 7B **instruction/chat** models from well-known publishers and pick three different profiles: the highest HellaSwag, the highest MMLU, and the best balanced score (the mean of the two). The three picks must come from **three different publishers**, so that we compare genuinely different models.

In [18]:
pool = lb[(lb.params_B <= 7) & lb.fullname.str.match(r"^(mistralai|microsoft|meta-llama|google|Qwen|openchat|HuggingFaceH4|tiiuae|01-ai|stabilityai)/")
          & lb.fullname.str.contains(r"instruct|chat|it$|beta|openchat", case=False)].copy()
pool["balance"] = (pool.HellaSwag + pool.MMLU) / 2
print(len(pool), "candidate chat/instruct models <= 7B")
display(pool.sort_values("HellaSwag", ascending=False)[COLS].head(5).round(1))
display(pool.sort_values("MMLU", ascending=False)[COLS].head(5).round(1))

28 candidate chat/instruct models <= 7B


,fullname,params_B,Architecture,license,ARC,HellaSwag,MMLU,Winogrande,GSM8K
6147,openchat/openchat-3.5-1210,7,MistralForCausalLM,apache-2.0,64.9,84.9,64.6,80.7,66.0
5821,mistralai/Mistral-7B-Instruct-v0.2,7,MistralForCausalLM,apache-2.0,63.1,84.9,60.8,77.2,40.0
6149,openchat/openchat_3.5,0,MistralForCausalLM,apache-2.0,63.8,84.8,65.0,80.7,26.6
1247,HuggingFaceH4/zephyr-7b-beta,7,MistralForCausalLM,mit,62.0,84.5,61.1,78.1,11.4
6145,openchat/openchat-3.5-0106,7,MistralForCausalLM,apache-2.0,66.0,82.9,65.0,81.8,68.2


,fullname,params_B,Architecture,license,ARC,HellaSwag,MMLU,Winogrande,GSM8K
5771,microsoft/Phi-3-mini-4k-instruct,3,Unknown,mit,63.0,80.6,69.1,72.4,74.5
5770,microsoft/Phi-3-mini-128k-instruct,3,Unknown,mit,63.1,80.1,68.7,72.8,69.5
6145,openchat/openchat-3.5-0106,7,MistralForCausalLM,apache-2.0,66.0,82.9,65.0,81.8,68.2
6149,openchat/openchat_3.5,0,MistralForCausalLM,apache-2.0,63.8,84.8,65.0,80.7,26.6
6147,openchat/openchat-3.5-1210,7,MistralForCausalLM,apache-2.0,64.9,84.9,64.6,80.7,66.0


In [19]:
pool["org"] = pool.fullname.str.split("/").str[0]
pick_hs = pool.sort_values("HellaSwag", ascending=False).iloc[0]
pick_mmlu = pool[pool.org != pick_hs.org].sort_values("MMLU", ascending=False).iloc[0]
pick_bal = pool[~pool.org.isin([pick_hs.org, pick_mmlu.org])].sort_values("balance", ascending=False).iloc[0]

use_cases = {pick_hs.fullname: "Customer-support / commonsense Q&A chatbot (strong everyday reasoning, permissive license)",
             pick_mmlu.fullname: "Academic question tutor / exam-prep assistant (best multi-subject knowledge for its size)",
             pick_bal.fullname: "General-purpose internal assistant (balanced commonsense + knowledge)"}
leaderboard_models = [{"model": r.fullname, "url": f"https://huggingface.co/{r.fullname}", "hellaswag": round(r.HellaSwag, 1),
                       "mmlu": round(r.MMLU, 1), "license": r.license, "ideal_use_case": use_cases[r.fullname]}
                      for r in [pick_hs, pick_mmlu, pick_bal]]
pd.DataFrame(leaderboard_models)

,model,url,hellaswag,mmlu,license,ideal_use_case
0,openchat/openchat-3.5-1210,https://huggingface.co/openchat/openchat-3.5-1210,84.9,64.6,apache-2.0,"Customer-support / commonsense Q&A chatbot (strong everyday reasoning, permissive license)"
1,microsoft/Phi-3-mini-4k-instruct,https://huggingface.co/microsoft/Phi-3-mini-4k-instruct,80.6,69.1,mit,Academic question tutor / exam-prep assistant (best multi-subject knowledge for its size)
2,mistralai/Mistral-7B-Instruct-v0.2,https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2,84.9,60.8,apache-2.0,General-purpose internal assistant (balanced commonsense + knowledge)


In [20]:
benchmark_table = "| Model Name | HellaSwag Score | MMLU Score | License Type | Ideal Use Case |\n| --- | --- | --- | --- | --- |\n" + "\n".join(
    f"| [{m['model']}]({m['url']}) | {m['hellaswag']} | {m['mmlu']} | {m['license']} | {m['ideal_use_case']} |" for m in leaderboard_models)
display(Markdown(benchmark_table))

| Model Name | HellaSwag Score | MMLU Score | License Type | Ideal Use Case |
| --- | --- | --- | --- | --- |
| [openchat/openchat-3.5-1210](https://huggingface.co/openchat/openchat-3.5-1210) | 84.9 | 64.6 | apache-2.0 | Customer-support / commonsense Q&A chatbot (strong everyday reasoning, permissive license) |
| [microsoft/Phi-3-mini-4k-instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) | 80.6 | 69.1 | mit | Academic question tutor / exam-prep assistant (best multi-subject knowledge for its size) |
| [mistralai/Mistral-7B-Instruct-v0.2](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2) | 84.9 | 60.8 | apache-2.0 | General-purpose internal assistant (balanced commonsense + knowledge) |

### Optional reflection
Benchmarks give **comparable, reproducible evidence** on the abilities a use case needs. Hype and parameter counts do not: in the table above, a 3.8B model beats several 7B models on MMLU, and a model fine-tuned for chat can lose math ability (OpenHermes-2.5: GSM8K 25 vs MetaMath-Mistral: 69 on the same Mistral base). Benchmarks also have limits: contamination, English-only tests, and leaderboard over-fitting. They should therefore shortlist models, and a test on **your own data and prompts** should make the final decision.

In [21]:
quiz_reflection = [
    {"question": "Your SaaS start-up needs a commercial chatbot with zero licensing risk. Llama-2-7B-chat (Llama 2 license) or Mistral-7B-Instruct (Apache 2.0)?",
     "answer": "Mistral-7B-Instruct: Apache 2.0 has no MAU cap, no acceptable-use restrictions beyond the law and no ban on training other models on its outputs."},
    {"question": "A school laptop has 8 GB RAM and no GPU. Is a 7B FP16 model realistic? What would you do?",
     "answer": "No: 7B in FP16 needs ~14 GB just for the weights. Use a ≤4B model (e.g. Phi-3-mini) in 4-bit GGUF (~2.4 GB) with llama.cpp."},
    {"question": "Which benchmark would you check first for a math tutor, and which for a multilingual NGO assistant?",
     "answer": "GSM8K (and MATH) for the tutor; FLORES-200 / multilingual MMLU and the languages listed on the model card for the NGO."},
]
pd.DataFrame(quiz_reflection)

,question,answer
0,Your SaaS start-up needs a commercial chatbot with zero licensing risk. Llama-2-7B-chat (Llama 2 license) or Mistral...,"Mistral-7B-Instruct: Apache 2.0 has no MAU cap, no acceptable-use restrictions beyond the law and no ban on training..."
1,A school laptop has 8 GB RAM and no GPU. Is a 7B FP16 model realistic? What would you do?,No: 7B in FP16 needs ~14 GB just for the weights. Use a ≤4B model (e.g. Phi-3-mini) in 4-bit GGUF (~2.4 GB) with lla...
2,"Which benchmark would you check first for a math tutor, and which for a multilingual NGO assistant?",GSM8K (and MATH) for the tutor; FLORES-200 / multilingual MMLU and the languages listed on the model card for the NGO.


## Exercise 6: Cloud vs. Local Deployment Plan
**Use:** cost, latency, scalability, security and maintenance trade-offs between local and cloud deployment; a timed 7B run.

In [22]:
pros_and_cons = [
    "✔️ Data privacy & control (local: data never leaves your machine, easier GDPR/HIPAA) vs. ❌ cloud: data sent to a third-party provider, needs a DPA and vendor trust",
    "✔️ No per-hour bill once you own the hardware (local) vs. ❌ high upfront hardware cost (a 24 GB GPU workstation costs about $2-4k) and no elasticity",
    "✔️ Easy access to powerful GPUs on demand (cloud: Colab, RunPod, A100/H100 by the hour) vs. ❌ local CPUs are slow (~3-4 tokens/s for a 7B 4-bit model, measured below)",
    "✔️ Low latency, no network hop and no cold starts once the model is loaded (local) vs. ❌ cloud: cold starts, network latency, session limits (Colab disconnects)",
    "✔️ Scalability & less maintenance (cloud: autoscaling, managed drivers/images) vs. ❌ local: you maintain drivers, updates and backups and cannot serve many users at once",
]
for b in pros_and_cons:
    print("-", b)

- ✔️ Data privacy & control (local: data never leaves your machine, easier GDPR/HIPAA) vs. ❌ cloud: data sent to a third-party provider, needs a DPA and vendor trust
- ✔️ No per-hour bill once you own the hardware (local) vs. ❌ high upfront hardware cost (a 24 GB GPU workstation costs about $2-4k) and no elasticity
- ✔️ Easy access to powerful GPUs on demand (cloud: Colab, RunPod, A100/H100 by the hour) vs. ❌ local CPUs are slow (~3-4 tokens/s for a 7B 4-bit model, measured below)
- ✔️ Low latency, no network hop and no cold starts once the model is loaded (local) vs. ❌ cloud: cold starts, network latency, session limits (Colab disconnects)
- ✔️ Scalability & less maintenance (cloud: autoscaling, managed drivers/images) vs. ❌ local: you maintain drivers, updates and backups and cannot serve many users at once


### Timed 7B run
- **No GPU** (this saved run, or Colab CPU): we run **Mistral-7B-Instruct-v0.3** in **4-bit GGUF (Q4_K_M, ~4.4 GB)** with llama.cpp. This is the "local deployment" scenario from Exercise 4.
- **With a GPU** (Colab T4: Runtime → Change runtime type), the second cell also runs the course snippet with 🤗 Transformers in FP16. We use the open `mistralai/Mistral-7B-Instruct-v0.3` because `meta-llama/Llama-2-7b-chat-hf` is gated: it requires accepting Meta's license and an HF token.

In [23]:
from llama_cpp import Llama
t0 = time.time()
llm = Llama.from_pretrained(repo_id="bartowski/Mistral-7B-Instruct-v0.3-GGUF", filename="Mistral-7B-Instruct-v0.3-Q4_K_M.gguf",
                            n_ctx=2048, n_threads=os.cpu_count(), verbose=False)
load_s = time.time() - t0
t0 = time.time()
resp = llm.create_chat_completion(messages=[{"role": "user", "content": "Hello, world! Introduce yourself in one sentence."}],
                                  max_tokens=50, temperature=0)
gen_s = time.time() - t0
print(resp["choices"][0]["message"]["content"])
n_tok = resp["usage"]["completion_tokens"]
print(f"\nLoad: {load_s:.1f}s | response: {gen_s:.1f}s for {n_tok} tokens ({n_tok/gen_s:.1f} tokens/s) on {os.cpu_count()} CPU threads")

 Hello! I am a large language model trained by Mistral AI, designed to assist and engage in conversations on a wide range of topics.

Load: 17.2s | response: 8.0s for 28 tokens (3.5 tokens/s) on 4 CPU threads


In [24]:
import torch
gpu_result = None
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    model_name = "mistralai/Mistral-7B-Instruct-v0.3"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")
    inputs = tokenizer("Hello, world!", return_tensors="pt").to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=50)
    print(tokenizer.decode(outputs[0]))
    gpu_result = time.time() - start
    print("Elapsed:", gpu_result, "seconds")
else:
    print("No GPU in this runtime: skipping the FP16 Transformers run (needs ~15 GB of GPU memory, e.g. a Colab T4).")

No GPU in this runtime: skipping the FP16 Transformers run (needs ~15 GB of GPU memory, e.g. a Colab T4).


In [25]:
colab_run = {
    "model_tested": "mistralai/Mistral-7B-Instruct-v0.3 (4-bit GGUF Q4_K_M via llama.cpp)" + (" + FP16 Transformers on GPU" if gpu_result else ""),
    "response_time_seconds": round(gen_s, 1) if not gpu_result else {"cpu_gguf": round(gen_s, 1), "gpu_fp16": round(gpu_result, 1)},
    "notes": f"Loaded in {load_s:.0f}s and generated {n_tok} tokens at {n_tok/gen_s:.1f} tok/s on {os.cpu_count()} CPU threads, "
             "with no GPU. That is usable for a single user or batch jobs, too slow for many concurrent users, which is where a cloud GPU wins.",
}
colab_run

{'model_tested': 'mistralai/Mistral-7B-Instruct-v0.3 (4-bit GGUF Q4_K_M via llama.cpp)',
 'response_time_seconds': 8.0,
 'notes': 'Loaded in 17s and generated 28 tokens at 3.5 tok/s on 4 CPU threads, with no GPU. That is usable for a single user or batch jobs, too slow for many concurrent users, which is where a cloud GPU wins.'}

### Colab / deployment experience (summary)
**Mistral-7B-Instruct-v0.3** in 4-bit GGUF (Q4_K_M, 4.4 GB) ran with **no GPU** on a 4-thread CPU with 15.7 GB of RAM. Loading took **17 s**, and the "Hello, world!" prompt got a **28-token answer in 8.0 s (≈ 3.5 tokens/s)**. This confirms Exercise 4: a 7B *quantized* model is usable on a modest Linux machine even though the "16 GB RAM / 40 GB disk" boxes are not ticked. FP16 (about 15 GB of weights) would need a GPU, so the cloud remains the better option for multi-user serving, longer outputs or unquantized models. On a Colab T4 runtime, the second cell runs the Transformers FP16 version for comparison.